# ✍️ 필적 분류 (Writer Identification) — ResNet18 Baseline

손글씨 이미지를 보고 **누가 쓴 글씨인지(작성자)** 를 맞추는 10-class 분류 문제입니다.

## 데이터 구조

```
12-aikuthon/
├── train/train/<writer>/*.png   # 학습 이미지 829장 (작성자 폴더명 = 클래스)
├── test/<id>.npz                # 암호화된 평가 데이터 490개 (512×512 RGB)
└── sample_submission.csv
```

- 작성자는 10명입니다: `csb, csh, ehj, lc, sj, sy, wj, yh, ys, ysh`
  폴더명을 정렬한 순서가 `test_loader` 의 `CLASSES` 순서와 같습니다.
- 학습 이미지는 **원본 해상도가 작성자마다 다릅니다** (폭 1008~4999px). 아래에서 폭을 통일합니다.
- 평가 데이터는 XOR 스트림 암호로 보호되어 있고, 원본 페이지에서 1024×1024 창을 잘라
  512×512로 리사이즈한 조각입니다. 페이지 전체가 아니라 **국소적인 획 특징**만 담겨 있습니다.

## 이 노트북이 하는 일

1. 페이지 이미지를 **패치(patch)** 로 잘라서 (페이지 전체를 224×224로 줄이면 획 정보가 다 뭉개지기 때문)
2. ImageNet 사전학습 **ResNet18**을 파인튜닝하고
3. 검증 패치 정확도가 가장 좋은 체크포인트를 저장합니다.

> ⚠️ **이 노트북은 학습까지만 다룹니다.** 암호화된 test 를 복호화해 추론하고
> `submission.csv` 를 만드는 코드는 들어 있지 않습니다. `test_loader` 의 `TestDS` 를 이용해
> 직접 구현해야 합니다. 복호화 결과를 화면에 출력하거나 파일로 저장하는 행위,
> 사람이 눈으로 보고 라벨을 부여하는 행위는 **규칙 위반이며 실격 사유**입니다.

## 평가 지표

리더보드는 **Macro F1** 입니다. 작성자별 F1 을 따로 구해 단순 평균하므로,
표본이 적은 작성자를 통째로 틀리면 그대로 점수에 반영됩니다.
아래에서 계산하는 **검증 패치 정확도는 참고용**일 뿐이며,
학습 데이터와 같은 촬영 세션에서 나온 값이라 리더보드 점수보다 훨씬 낙관적입니다.

## 필요 패키지

```bash
pip install torch torchvision pillow numpy matplotlib
```


## 1. 임포트 & 시드 고정

In [ ]:
import os
import random
import sys
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights

# 재현성을 위해 시드 고정 (cudnn 완전 결정론까지는 강제하지 않음 — baseline 수준에서는 충분)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Colab GPU 런타임이면 cuda, Apple Silicon 로컬이면 mps, 그 외 cpu
DEVICE = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"device = {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. 설정

경로와 하이퍼파라미터를 한곳에 모아 둡니다. 데이터 위치가 다르면 `DATA_DIR` 만 바꾸면 됩니다.

In [ ]:
# ======================== 경로 ========================
# 대회 데이터셋 구조
#   12-aikuthon/train/train/<writer>/*.png   학습 이미지 829장
#   12-aikuthon/test/<id>.npz                암호화 평가 데이터 490개
#   12-aikuthon/sample_submission.csv
ROOT      = Path.cwd()
DATA_DIR  = ROOT / "12-aikuthon"
TRAIN_DIR = DATA_DIR / "train" / "train"   # <writer>/*.png
TEST_ROOT = DATA_DIR                       # TestDS 가 {TEST_ROOT}/test/<id>.npz 를 찾음

CACHE_DIR = ROOT / "patches_cache"         # 잘라낸 패치를 저장(캐시)하는 폴더 — 두 번째 실행부터는 재사용
CKPT_PATH = ROOT / "baseline_resnet18.pt"

# ======================== 패치 추출 파라미터 ========================
PAGE_WIDTH = 1280   # 페이지 가로폭을 이 값으로 통일 (해상도가 제각각이라 스케일을 맞춤)
PATCH = 256         # 통일된 페이지 위에서 잘라내는 패치 한 변의 크기
STRIDE = 192        # 슬라이딩 윈도우 간격 (25% 겹침)
MIN_INK_FRAC = 0.01 # 패치 내 '잉크' 픽셀 비율이 이 값 미만이면 (= 거의 빈 종이면) 버림
SAVE_SIZE = 224     # 저장 크기 = ResNet18 입력 크기

# ======================== 학습 하이퍼파라미터 ========================
EPOCHS = 6
BATCH_SIZE = 128
LR = 3e-4
WEIGHT_DECAY = 1e-4
VAL_FRAC = 0.15     # 검증용으로 떼어 놓는 "페이지" 비율 (패치가 아니라 페이지 단위! 아래 설명 참고)

# Windows / macOS 는 multiprocessing 기본값이 spawn 이라, 노트북에서 정의한 Dataset 을
# 워커가 언피클하지 못해 깨집니다. Linux(Colab) 에서만 워커를 씁니다.
NUM_WORKERS = 0 if (os.name == "nt" or sys.platform == "darwin") else 4

print(f"TRAIN_DIR = {TRAIN_DIR}")
print(f"TEST_ROOT = {TEST_ROOT}")

## 3. 데이터 인벤토리

작성자별로 어떤 폴더에 몇 장이 있는지 확인하고, **현재 모드에서 분류 대상이 되는 작성자 목록(`WRITERS`)** 을 확정합니다.
학습 페이지가 한 장도 없는 작성자는 클래스에 포함할 수 없으므로 제외합니다.

In [ ]:
IMG_EXTS = (".png", ".jpg", ".jpeg")

def list_pages(d: Path):
    """폴더 안의 페이지 파일(.png/.jpg/.jpeg/.pdf) 목록. 그 외 파일(csv 등)은 무시."""
    if not d.is_dir():
        return []
    return sorted(f for f in d.iterdir()
                  if f.suffix.lower() in IMG_EXTS + (".pdf",))

# 폴더명이 곧 클래스입니다. 정렬 순서가 test_loader 의 CLASSES 와 동일합니다.
WRITERS = sorted(p.name for p in TRAIN_DIR.iterdir() if p.is_dir())
W2I = {w: i for i, w in enumerate(WRITERS)}   # writer 이름 -> 클래스 인덱스

print(f"{'writer':6s} | {'pages':>6s}")
print("-" * 17)
total = 0
for w in WRITERS:
    n = len(list_pages(TRAIN_DIR / w))
    total += n
    print(f"{w:6s} | {n:6d}")
print("-" * 17)
print(f"{'total':6s} | {total:6d}")

print(f"\n분류 대상 작성자 ({len(WRITERS)}명): {WRITERS}")

## 4. 페이지 로딩 & 패치 추출

**왜 패치로 자르나?** 페이지 전체를 224×224로 리사이즈하면 글씨 획이 몇 픽셀 수준으로 뭉개져서
필적의 미세한 특징(획의 꺾임, 삐침, 자간 등)이 사라집니다. 그래서 페이지를 일정 크기 패치로 잘라
패치 단위로 학습/예측하고, 페이지의 최종 예측은 패치 예측들의 평균으로 만듭니다.

처리 과정:
1. **로딩** — PNG/JPG는 PIL로, PDF는 PyMuPDF로 렌더링. 투명 배경(RGBA)은 흰색 위에 합성.
2. **폭 통일** — 페이지 가로폭을 `PAGE_WIDTH`로 리사이즈 (기기마다 해상도가 제각각이므로).
3. **슬라이딩 윈도우** — `PATCH`×`PATCH` 크기로 `STRIDE` 간격으로 자름.
4. **빈 패치 제거** — 배경(종이)보다 충분히 어두운 픽셀을 '잉크'로 보고, 잉크 비율이
   `MIN_INK_FRAC` 미만인 패치(거의 빈 종이)는 버림. 다크모드 스크린샷(어두운 배경 + 밝은 글씨)은
   반전해서 동일하게 처리.

잘라낸 패치는 `patches_cache/`에 저장해 두므로, **모드를 바꿔 다시 실행해도 추출은 한 번만** 합니다.

> 💡 여기서는 의도적으로 리사이즈/잘라내기 같은 일반적인 전처리만 합니다.
> 테스트 데이터 분포를 안다고 가정한 전처리(특정 도메인에 맞춘 정규화 등)는 baseline에 넣지 않았습니다 —
> 그런 아이디어는 여러분이 직접 실험해 볼 부분입니다.

In [ ]:
def load_page(path: Path) -> Image.Image:
    """페이지 파일을 RGB PIL 이미지로 로딩. PDF는 렌더링, RGBA는 흰 배경에 합성."""
    if path.suffix.lower() == ".pdf":
        import fitz  # PyMuPDF (pip install pymupdf)
        doc = fitz.open(path)
        # PDF 좌표계는 72dpi 기준이라 그대로 렌더링하면 저해상도 -> PAGE_WIDTH에 맞게 확대 렌더링
        pages = []
        for pg in doc:
            zoom = PAGE_WIDTH / max(1.0, pg.rect.width)
            pix = pg.get_pixmap(matrix=fitz.Matrix(zoom, zoom))
            pages.append(Image.frombytes("RGB", (pix.width, pix.height), pix.samples))
        doc.close()
        if len(pages) == 1:
            return pages[0]
        # 여러 페이지짜리 PDF면 세로로 이어붙여 하나의 긴 페이지로 취급
        w = max(p.width for p in pages)
        canvas = Image.new("RGB", (w, sum(p.height for p in pages)), (255, 255, 255))
        y = 0
        for p in pages:
            canvas.paste(p, (0, y)); y += p.height
        return canvas

    img = Image.open(path)
    if img.mode in ("RGBA", "LA", "P"):
        # 투명 배경을 검정으로 뭉개지 않도록 흰색 캔버스 위에 합성
        img = img.convert("RGBA")
        canvas = Image.new("RGBA", img.size, (255, 255, 255, 255))
        img = Image.alpha_composite(canvas, img)
    return img.convert("RGB")


def extract_patches(page_path: Path, out_dir: Path) -> int:
    """페이지 한 장 -> 잉크가 있는 패치들을 out_dir에 저장. 저장한 패치 수를 반환."""
    img = load_page(page_path)

    # 1) 가로폭 통일
    w, h = img.size
    scale = PAGE_WIDTH / w
    img = img.resize((PAGE_WIDTH, max(1, round(h * scale))), Image.LANCZOS)

    # 2) 잉크 마스크 계산 (그레이스케일에서 배경보다 충분히 어두운 픽셀 = 잉크)
    gray = np.asarray(img.convert("L")).astype(np.int16)
    bg = np.median(gray)                 # 배경 밝기 추정 (페이지 대부분은 종이이므로 중앙값 ≈ 배경)
    if bg < 128:                         # 다크모드(어두운 배경 + 밝은 글씨)면 반전
        gray = 255 - gray
        bg = np.median(gray)
    ink = gray < max(40, bg - 60)        # 배경보다 60 이상 어두우면 잉크로 간주

    # 페이지가 패치보다 작으면 흰색으로 패딩
    arr = np.asarray(img)
    H, W = arr.shape[:2]
    if H < PATCH or W < PATCH:
        pad_h, pad_w = max(0, PATCH - H), max(0, PATCH - W)
        arr = np.pad(arr, ((0, pad_h), (0, pad_w), (0, 0)), constant_values=255)
        ink = np.pad(ink, ((0, pad_h), (0, pad_w)), constant_values=False)
        H, W = arr.shape[:2]

    # 3) 슬라이딩 윈도우로 자르면서 4) 빈 패치는 버림
    out_dir.mkdir(parents=True, exist_ok=True)
    n = 0
    for y in range(0, H - PATCH + 1, STRIDE):
        for x in range(0, W - PATCH + 1, STRIDE):
            if ink[y:y + PATCH, x:x + PATCH].mean() < MIN_INK_FRAC:
                continue  # 잉크가 거의 없는 패치(빈 종이)는 스킵
            tile = Image.fromarray(arr[y:y + PATCH, x:x + PATCH])
            tile = tile.resize((SAVE_SIZE, SAVE_SIZE), Image.LANCZOS)
            tile.save(out_dir / f"{page_path.stem}_y{y:05d}_x{x:05d}.png")
            n += 1
    return n

In [ ]:
# 학습 페이지 전체에 대해 패치 추출을 실행합니다.
# 캐시 구조: patches_cache/<writer>/<page_stem>/  (페이지당 폴더 하나)
# 페이지 폴더가 이미 존재하면 추출이 끝난 것으로 보고 건너뜁니다.
#
# 평가 데이터는 암호화되어 있어 여기서 다루지 않습니다. 복호화 결과를 파일로 저장하는 것은
# 대회 규칙 위반이므로, test 는 추론 시점에 메모리에서만 처리해야 합니다.
total_pages = total_patches = 0
for w in WRITERS:
    for page in list_pages(TRAIN_DIR / w):
        out_dir = CACHE_DIR / w / page.stem
        if not out_dir.is_dir():                     # 캐시에 없을 때만 추출
            extract_patches(page, out_dir)
        total_pages += 1
        total_patches += sum(1 for _ in out_dir.glob("*.png"))
print(f"총 {total_pages}개 페이지 -> {total_patches}개 패치 (캐시: {CACHE_DIR.name}/)")

## 5. 패치 샘플 확인

잘라낸 패치가 어떻게 생겼는지 눈으로 확인해 봅니다. 데이터를 직접 보는 습관은 언제나 중요합니다!

In [ ]:
import matplotlib.pyplot as plt

# 작성자별로 패치 하나씩 샘플링
fig, axes = plt.subplots(2, (len(WRITERS) + 1) // 2, figsize=(2.2 * ((len(WRITERS) + 1) // 2), 5))
axes = np.atleast_1d(axes).ravel()
for ax, w in zip(axes, WRITERS):
    patches = sorted((CACHE_DIR / w).rglob("*.png"))
    sample = patches[len(patches) // 2] if patches else None   # 중간쯤의 패치 하나
    if sample:
        ax.imshow(Image.open(sample), cmap="gray")
        ax.set_title(w, fontsize=10)
    ax.axis("off")
for ax in axes[len(WRITERS):]:
    ax.axis("off")
plt.suptitle("train patches per writer")
plt.tight_layout()
plt.show()

## 6. Dataset & 학습/검증 분할

⚠️ **분할은 반드시 "페이지 단위"로!** 같은 페이지에서 나온 패치들은 서로 매우 비슷합니다
(겹치는 영역도 있고, 같은 펜·같은 종이·같은 날 쓴 글씨). 패치 단위로 무작위 분할하면
같은 페이지의 패치가 train과 val 양쪽에 들어가서 **검증 정확도가 뻥튀기(데이터 누수)** 됩니다.
그래서 작성자별로 페이지의 15%를 통째로 검증셋에 배정합니다.

정규화는 ImageNet 사전학습 백본의 표준 통계(mean/std)를 그대로 씁니다 —
사전학습 가중치를 쓸 때의 관례이며, 이 대회 데이터의 통계에 맞춘 값이 아닙니다.

In [ ]:
# ImageNet 사전학습 모델의 표준 정규화 통계 (torchvision 공식 값)
IMAGENET_NORM = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

# 학습용 증강: 가벼운 기하/밝기 변형만.
# 좌우반전(hflip)은 글자가 뒤집혀 부자연스러우므로 쓰지 않습니다.
train_tf = transforms.Compose([
    transforms.RandomRotation(5, fill=255),                                  # 살짝 기울이기 (여백은 흰색)
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0), ratio=(0.9, 1.1)),   # 확대/축소 + 랜덤 크롭
    transforms.ColorJitter(brightness=0.2, contrast=0.2),                    # 조명/스캔 밝기 차이 흉내
    transforms.ToTensor(),
    IMAGENET_NORM,
])
# 평가용: 변형 없이 그대로 (패치는 이미 224×224로 저장돼 있음)
eval_tf = transforms.Compose([transforms.ToTensor(), IMAGENET_NORM])


class PatchDataset(Dataset):
    """(패치 파일 경로, 클래스 인덱스) 리스트를 받아 이미지를 로딩하는 단순 Dataset."""
    def __init__(self, files, labels, tf):
        self.files, self.labels, self.tf = files, labels, tf

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        img = Image.open(self.files[i]).convert("RGB")
        return self.tf(img), self.labels[i]


def build_train_val():
    """작성자별로 페이지를 섞은 뒤, 페이지의 VAL_FRAC 만큼을 통째로 검증셋으로."""
    tr_f, tr_l, va_f, va_l = [], [], [], []
    for w in WRITERS:
        page_dirs = sorted(p for p in (CACHE_DIR / w).glob("*") if p.is_dir())
        if not page_dirs:
            continue
        rng = random.Random(SEED)          # 실행마다 같은 페이지가 val 에 가도록 시드 고정
        rng.shuffle(page_dirs)
        n_val = max(1, round(len(page_dirs) * VAL_FRAC))
        for k, pd in enumerate(page_dirs):
            files = sorted(pd.glob("*.png"))
            dst_f, dst_l = (va_f, va_l) if k < n_val else (tr_f, tr_l)
            dst_f.extend(files)
            dst_l.extend([W2I[w]] * len(files))
    return tr_f, tr_l, va_f, va_l


train_files, train_labels, val_files, val_labels = build_train_val()
print(f"train patches = {len(train_files)},  val patches = {len(val_files)}")

# 클래스(작성자)별 학습 패치 수 — 불균형이 꽤 있습니다. baseline은 그대로 학습하지만,
# 가중 샘플링(WeightedRandomSampler)이나 클래스 가중 loss는 좋은 개선 포인트입니다.
cnt = Counter(train_labels)
for w in WRITERS:
    print(f"  {w:6s}: {cnt[W2I[w]]:6d} patches")

train_loader = DataLoader(PatchDataset(train_files, train_labels, train_tf),
                          batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))
val_loader = DataLoader(PatchDataset(val_files, val_labels, eval_tf),
                        batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))

## 7. 모델

ImageNet 사전학습 **ResNet18**의 마지막 fully-connected 층만 작성자 수에 맞게 교체합니다.
백본 전체를 파인튜닝합니다 (freeze 없음 — 데이터가 자연 이미지와 많이 달라서 전체 튜닝이 유리).

In [ ]:
model = resnet18(weights=ResNet18_Weights.DEFAULT)          # ImageNet 사전학습 가중치
model.fc = nn.Linear(model.fc.in_features, len(WRITERS))    # 분류 헤드 교체
model = model.to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)  # LR을 점차 감소
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)       # 라벨 스무딩으로 과신 방지
scaler = torch.amp.GradScaler(enabled=(DEVICE == "cuda"))   # mixed precision (GPU에서 학습 가속)

print(f"ResNet18, {len(WRITERS)} classes, "
      f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M params")

## 8. 학습

에폭마다 검증 패치 정확도를 확인하고, **가장 좋은 검증 성능의 체크포인트**를 저장합니다.
(마지막 에폭이 항상 최선은 아니므로.)

In [ ]:
@torch.no_grad()
def eval_patch_acc(model, loader):
    """패치 단위 정확도 (검증용 지표)."""
    model.eval()
    correct = total = 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.autocast(DEVICE, enabled=(DEVICE == "cuda")):
            out = model(x)
        correct += (out.argmax(1) == y).sum().item()
        total += y.numel()
    return correct / max(1, total)


best_val = 0.0
for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = seen = 0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(DEVICE, enabled=(DEVICE == "cuda")):
            loss = criterion(model(x), y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * y.numel()
        seen += y.numel()
    scheduler.step()

    val_acc = eval_patch_acc(model, val_loader)
    flag = ""
    if val_acc >= best_val:                      # 최고 성능 갱신 시 체크포인트 저장
        best_val = val_acc
        torch.save({"model": model.state_dict(), "writers": WRITERS}, CKPT_PATH)
        flag = "  <- best, saved"
    print(f"epoch {epoch}/{EPOCHS}  train_loss={running_loss / seen:.4f}"
          f"  val_patch_acc={val_acc:.4f}{flag}", flush=True)

print(f"\nbest val patch acc = {best_val:.4f}  (checkpoint: {CKPT_PATH.name})")

## 9. 다음 단계: 개선 아이디어

이 baseline에서 출발해 볼 만한 방향들입니다.

**데이터/전처리**
- 클래스 불균형 처리: `WeightedRandomSampler`, 클래스 가중 loss
- 패치 크기·간격·잉크 임계값 튜닝, 멀티스케일 패치
- 획 두께/배경을 무시하도록 만드는 증강 (모폴로지 연산으로 획 굵기 랜덤화, 이진화 등)
- 배경(격자, 그림자, 압축 아티팩트)을 분리/제거하는 전처리

**모델/학습**
- 더 큰 백본 (ResNet50, ConvNeXt, ViT ...), 더 긴 학습 + 강한 증강
- 분류 대신 metric learning (ArcFace, triplet loss) — "필적 임베딩"을 배우면 학습에 없던 작성자 대응도 가능
- 자기지도 사전학습 (본 대회 데이터로 SimCLR/MAE 등)

**추론**
- Test-Time Augmentation (여러 스케일/회전으로 예측 후 평균)
- 패치 확률 평균 대신 다른 집계 방식 (신뢰도 가중 평균, 다수결, top-k 평균 등)

**분석**
- 오답 페이지를 직접 열어 보기 — 어떤 도메인/언어에서 틀리는지 파악이 개선의 첫걸음
- Grad-CAM으로 모델이 글씨를 보는지, 배경(종이, 격자, 압축 아티팩트)을 보는지 확인